# TP3 — Análise de Dados Assistida por IA

**CloudDesk — Base de Tickets de Suporte**  
**Aluno: Guilherme Reis de Jesus Santos**  
**Disciplina: DR1 — Engenharia de Dados com pandas**

---

## Setup — Recarregamento e consolidação da base (mesmo pipeline do TP1)

Recarregamento dos três arquivos brutos e repetição das etapas de limpeza e merge do TP1, chegando ao mesmo `df_consolidado`.

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta

# Regeneração dos arquivos brutos (TP1)
rng = np.random.default_rng(42)
n = 4000
start_date = datetime(2026, 1, 5)

ticket_ids = [f"TCK-{i:05d}" for i in range(1, n + 1)]
cliente_ids = [f"CLI-{i:03d}" for i in rng.integers(1, 181, n)]
dias_offset = np.sort(rng.integers(0, 180, n))
datas = [(start_date + timedelta(days=int(d))).strftime("%Y-%m-%d") for d in dias_offset]
canais = rng.choice(["email", "chat", "telefone"], size=n, p=[0.45, 0.35, 0.20])
categorias = rng.choice(["bug", "duvida_de_uso", "cobranca", "elogio"], size=n, p=[0.35, 0.30, 0.25, 0.10])
prioridades = rng.choice(["baixa", "media", "alta", "critica"], size=n, p=[0.30, 0.35, 0.25, 0.10])
tempo_resposta = np.clip(rng.gamma(shape=2.0, scale=2.2, size=n), 0.2, 48)
tempo_resposta[canais == "chat"] *= 0.6
tempo_resolucao = np.clip(1.8 * tempo_resposta + 3 + rng.normal(0, 1.5, n), 0.5, 120)
satisfacao = 5 - (tempo_resposta / 6) + rng.normal(0, 0.6, n)
satisfacao[canais == "chat"] -= 0.6
satisfacao = np.clip(np.round(satisfacao), 1, 5)

df = pd.DataFrame({
    "ticket_id": ticket_ids,
    "cliente_id": cliente_ids,
    "data_abertura": datas,
    "canal": canais,
    "categoria": categorias,
    "prioridade": prioridades,
    "tempo_resposta_horas": tempo_resposta,
    "tempo_resolucao_horas": tempo_resolucao,
    "satisfacao_cliente": satisfacao,
})

df.loc[rng.choice(n, size=int(n * 0.08), replace=False), "tempo_resposta_horas"] = np.nan
df.loc[rng.choice(n, size=int(n * 0.10), replace=False), "satisfacao_cliente"] = np.nan

def fmt_virgula(v):
    return "" if pd.isna(v) else f"{v:.2f}".replace(".", ",")

df["tempo_resposta_horas"] = df["tempo_resposta_horas"].apply(fmt_virgula)

duplicatas = df.sample(frac=0.02, random_state=42)
df_csv = pd.concat([df, duplicatas], ignore_index=True)
df_csv.to_csv("tickets_suporte.csv", index=False)

clientes_unicos = sorted(set(cliente_ids))
planos = pd.DataFrame({
    "cliente_id": clientes_unicos,
    "valor_plano": rng.choice([99.0, 199.0, 349.0, 599.0], size=len(clientes_unicos), p=[0.4, 0.3, 0.2, 0.1]),
    "segmento": rng.choice(["starter", "growth", "enterprise"], size=len(clientes_unicos), p=[0.5, 0.35, 0.15]),
})
planos.to_excel("planos_clientes.xlsx", index=False)

chat_ids = df.loc[df["canal"] == "chat", "ticket_id"].tolist()
triagem = [
    {"ticket_id": tid,
     "intencao_triagem": str(rng.choice(["duvida", "reclamacao", "solicitacao", "elogio"])),
     "confianca_bot": round(float(rng.uniform(0.55, 0.98)), 2)}
    for tid in chat_ids
]
with open("chatbot_triagem.json", "w", encoding="utf-8") as f:
    json.dump(triagem, f, ensure_ascii=False, indent=2)

print("Arquivos gerados: tickets_suporte.csv, planos_clientes.xlsx, chatbot_triagem.json")

In [ ]:
# Pipeline idêntico ao TP1: carga + conversão de tipos + nulos + duplicatas + merge 
df_tickets = pd.read_csv("tickets_suporte.csv")
df_planos  = pd.read_excel("planos_clientes.xlsx")
df_chatbot = pd.read_json("chatbot_triagem.json")

# Conversão de tipos (Exercício 3 do TP1)
df_tickets["data_abertura"] = pd.to_datetime(df_tickets["data_abertura"])
df_tickets["tempo_resposta_horas"] = (
    df_tickets["tempo_resposta_horas"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .replace("", float("nan"))
    .astype(float)
)
df_tickets["prioridade"] = df_tickets["prioridade"].astype("category")
df_tickets["canal"]      = df_tickets["canal"].astype("category")

# Nulos: mediana por categoria (Exercício 6 do TP1)
df_tickets["tempo_resposta_horas"] = df_tickets.groupby("categoria")["tempo_resposta_horas"].transform(
    lambda x: x.fillna(x.median())
)

# Duplicatas (Exercício 7 do TP1)
df_tickets = df_tickets.drop_duplicates(subset=["ticket_id"], keep="first")

# Merge (Exercício 8 do TP1)
df_consolidado = (
    df_tickets
    .merge(df_planos,  on="ticket_id" if False else "cliente_id", how="left")
    .merge(df_chatbot, on="ticket_id", how="left")
)

print(f"Shape final: {df_consolidado.shape}")
print(f"Colunas: {df_consolidado.columns.tolist()}")
print(f"Nulos restantes:\n{df_consolidado.isnull().sum()[df_consolidado.isnull().sum() > 0]}")

---

## Exercício 1 — Limpeza com LLM como copiloto

### Prompt enviado ao Claude (Anthropic)

```
Tenho um DataFrame pandas chamado df_tickets com as seguintes colunas relevantes:
- ticket_id (str): identificador único de cada ticket, formato "TCK-NNNNN"
- categoria (str/category): valores possíveis são "bug", "duvida_de_uso", "cobranca", "elogio"
- tempo_resposta_horas (float): pode ter NaN em ~8% das linhas

Regras de negócio:
1. Preencha os NaN de tempo_resposta_horas com a MEDIANA calculada dentro de cada
   grupo de categoria (mediana por categoria, não mediana global).
2. Remova linhas com ticket_id duplicado, mantendo apenas a PRIMEIRA ocorrência.

Gere apenas o código pandas que executa essas duas operações, sem explicações longas.
```

### Resposta do LLM

In [ ]:
# CÓDIGO GERADO PELO LLM

# 1. Preencher NaN de tempo_resposta_horas com a mediana por categoria
df_tickets["tempo_resposta_horas"] = df_tickets.groupby("categoria")["tempo_resposta_horas"].transform(
    lambda x: x.fillna(x.median())
)

# 2. Remover duplicatas de ticket_id, mantendo a primeira ocorrência
df_tickets = df_tickets.drop_duplicates(subset=["ticket_id"], keep="first")

# CÓDIGO GERADO PELO LLM

# Verificação
print("NaN em tempo_resposta_horas após preenchimento:", df_tickets["tempo_resposta_horas"].isna().sum())
print("Linhas restantes após remoção de duplicatas:", len(df_tickets))
print("ticket_ids únicos:", df_tickets["ticket_id"].nunique())

### Resultado vs TP1

Saiu 0 NaN e 4.000 linhas — igual ao feito manualmente no TP1. Código correto.

---

## Exercício 2 — Revisão crítica do código gerado

### Três coisas pra checar antes de confiar no código

**1. Mediana por categoria vs. mediana global**  
O LLM usou `groupby("categoria").transform(lambda x: x.fillna(x.median()))`, que calcula a mediana dentro de cada grupo. Precisa confirmar que não foi `df_tickets["tempo_resposta_horas"].median()` (global), que seria mais fácil de escrever mas errado.

**2. Ordem das operações**  
O código preenche NaN antes de remover duplicatas. Funciona, mas se a ordem fosse invertida a mediana por categoria mudaria um pouco — vale confirmar que é intencional.

**3. Coluna `categoria` como dtype `category`**  
Com esse dtype, o `groupby` pode incluir categorias vazias dependendo da versão do pandas. O `.transform()` esconde isso na prática, mas é bom verificar que toda categoria com NaN tem pelo menos um valor não-nulo (o que é o caso aqui com 4.000 linhas).

In [ ]:
# Teste em subconjunto conhecido: verificar que a mediana é por categoria, não global

# Recarregar estado bruto para testar isoladamente
df_test = pd.read_csv("tickets_suporte.csv")
df_test["tempo_resposta_horas"] = (
    df_test["tempo_resposta_horas"]
    .astype(str).str.replace(",", ".", regex=False)
    .replace("", float("nan")).astype(float)
)

# Mediana global x mediana por categoria
mediana_global = df_test["tempo_resposta_horas"].median()
medianas_cat   = df_test.groupby("categoria")["tempo_resposta_horas"].median()

print(f"Mediana global: {mediana_global:.4f}")
print("\nMediana por categoria:")
print(medianas_cat)

# As medianas por categoria diferem da mediana global?
print("\nDiferença real entre mediana global e por categoria:")
for cat, med in medianas_cat.items():
    print(f"  {cat}: {med:.4f} (diff vs global: {med - mediana_global:+.4f})")

### Erros encontrados e correção

O código não tinha erros. A estratégia `groupby + transform + fillna(median)` estava certa.

O teste acima confirma que as medianas por categoria diferem da global — então usar a global teria imputado valores ligeiramente diferentes.

**Diferença LLM vs. versão final:** nenhuma.

**Observação para produção:** se alguma categoria tivesse *todos* os valores como NaN, `x.median()` retornaria NaN e os registros ficariam sem preenchimento. Versão defensiva:

```python
# fallback para a mediana global se alguma categoria estiver toda nula
mediana_global = df_tickets["tempo_resposta_horas"].median()
df_tickets["tempo_resposta_horas"] = (
    df_tickets.groupby("categoria")["tempo_resposta_horas"]
    .transform(lambda x: x.fillna(x.median()))
    .fillna(mediana_global)
)
```

---

## Exercício 3 — Formulação de perguntas analíticas com apoio de IA

### Prompt enviado ao LLM

```
Tenho um dataset de tickets de suporte da CloudDesk com as seguintes colunas:
ticket_id, cliente_id, data_abertura (datetime), canal (email/chat/telefone),
categoria (bug/duvida_de_uso/cobranca/elogio), prioridade (baixa/media/alta/critica),
tempo_resposta_horas (float), tempo_resolucao_horas (float),
satisfacao_cliente (float 1-5, ~10% NaN), valor_plano (99/199/349/599),
segmento (starter/growth/enterprise), intencao_triagem (NaN para não-chat),
confianca_bot (float, NaN para não-chat).

Quero entender quais fatores influenciam a satisfação do cliente.
Gere uma lista de 8 perguntas analíticas respondíveis com esses dados.
```

### Resposta do LLM — perguntas geradas

1. Tickets abertos pelo canal "chat" têm satisfação média menor que tickets abertos por "email"?
2. A satisfação varia significativamente entre as categorias de ticket (bug, cobrança, dúvida de uso, elogio)?
3. Existe correlação entre `tempo_resposta_horas` e `satisfacao_cliente`?
4. Clientes de planos mais caros (enterprise/599) têm satisfação maior que os de planos básicos (starter/99)?
5. A satisfação varia ao longo do período coberto pela base (há tendência temporal)?
6. Tickets de prioridade "crítica" têm tempo de resolução maior e satisfação menor?
7. A intenção detectada pelo chatbot (reclamação vs. elogio) tem correlação com satisfação final?
8. A sazonalidade (dia da semana de abertura) influencia a satisfação?

### Seleção das três perguntas mais respondíveis

**Pergunta 1 (selecionada):** Chat vs. email na satisfação — as duas colunas estão completas e é a comparação central dos exercícios 5 e 6.

**Pergunta 3 (selecionada):** Correlação entre tempo de resposta e satisfação — duas colunas numéricas limpas, direto com `corr()` ou pearsonr.

**Pergunta 6 (selecionada):** Tickets críticos têm resolução mais lenta e satisfação menor? — `prioridade`, `tempo_resolucao_horas` e `satisfacao_cliente` disponíveis, groupby simples.

### Perguntas descartadas

**Pergunta 7:** `intencao_triagem` só existe para o canal chat (~35% dos dados). Comparar satisfação por intenção misturaria o efeito do canal com o da intenção — não dá pra separar os dois com esses dados.

**Pergunta 8:** Dia da semana exigiria distribuição uniforme de volume por dia ao longo dos 180 dias, o que não é garantido na base. Precisaria de dado de staffing que não existe aqui.

---

## Exercício 4 — Análise descritiva com código gerado por LLM

### Pergunta 1: Satisfação média por canal

In [ ]:
# CÓDIGO GERADO PELO LLM — Pergunta 1
resumo_canal = (
    df_consolidado
    .groupby("canal", observed=True)["satisfacao_cliente"]
    .agg(media="mean", mediana="median", n="count", desvio="std")
    .round(3)
    .sort_values("media")
)
print(resumo_canal)

**Resultado:** chat tem a menor satisfação média dos três canais. Clientes atendidos por chat estão consistentemente menos satisfeitos.

### Pergunta 3: Correlação entre tempo de resposta e satisfação

In [ ]:
# CODIGO GERADO PELO LLM — Pergunta 3
df_valido = df_consolidado.dropna(subset=["satisfacao_cliente"])

corr_pearson = df_valido[["tempo_resposta_horas", "satisfacao_cliente"]].corr().iloc[0, 1]
print(f"Correlação de Pearson (tempo_resposta_horas × satisfacao_cliente): {corr_pearson:.4f}")

# Agregação por quartil de tempo de resposta
df_valido = df_valido.copy()
df_valido["quartil_tempo"] = pd.qcut(df_valido["tempo_resposta_horas"], q=4,
                                      labels=["Q1 (rápido)", "Q2", "Q3", "Q4 (lento)"])
print("\nSatisfação média por quartil de tempo de resposta:")
print(
    df_valido.groupby("quartil_tempo", observed=True)["satisfacao_cliente"]
    .agg(media="mean", n="count")
    .round(3)
)

**Resultado:** correlação negativa — quanto mais rápido o atendimento, maior a satisfação. Clientes do quartil mais lento têm nota nitidamente inferior ao quartil mais rápido. SLA de tempo de resposta é alavanca direta de satisfação.

### Pergunta 6: Satisfação e tempo de resolução por prioridade

In [ ]:
# CÓDIGO GERADO PELO LLM — Pergunta 6
resumo_prior = (
    df_consolidado
    .groupby("prioridade", observed=True)
    .agg(
        media_satisfacao=("satisfacao_cliente", "mean"),
        media_resolucao=("tempo_resolucao_horas", "mean"),
        n=("ticket_id", "count"),
    )
    .round(3)
)
# Ordenar pela severidade lógica
ordem = ["baixa", "media", "alta", "critica"]
resumo_prior = resumo_prior.reindex([p for p in ordem if p in resumo_prior.index])
print(resumo_prior)

**Resultado:** tickets críticos têm tempo de resolução mais alto, mas a queda na satisfação não é proporcional — provavelmente porque clientes que abrem críticos já esperam demora. O ponto de atenção é comunicação proativa, não só o SLA.

---

## Exercício 5 — Comparação por intervalo de confiança (bootstrap)

### Item 1 — Sugestão inicial do LLM

**Prompt:** "Como comparar a satisfação média entre chat e email estatisticamente?"

**LLM sugeriu:** teste t de Student para amostras independentes (`scipy.stats.ttest_ind`), p-valor < 0,05.

### Item 2 — Explicação do p-valor pelo LLM e validação

**Prompt:** "Explica o que o p-valor representa, o que um analista deve e não deve concluir, e como se relaciona com a distribuição amostral."

**Resposta do LLM:**  
> "O p-valor é a probabilidade de observar uma diferença tão extrema (ou mais) assumindo que H0 é verdadeira. p < 0,05 indica que a diferença é improvável sob H0, mas não prova que H1 é verdadeira nem que a diferença tem relevância prática. Ele vem da distribuição amostral da estatística de teste."

**Validação:**

-  Correto: p-valor é probabilidade sob H0, não probabilidade de H0 ser verdadeira.
-  Correto: p < 0,05 não significa relevância prática.
-  Omitiu: com amostras grandes como as deste dataset (~1.400 chat, ~1.800 email), qualquer diferença mínima gera p < 0,05. O teste não diz nada sobre o tamanho do efeito.
-  Omitiu: p-valor não informa *quanto* as médias diferem, só se a diferença é inconsistente com H0. IC comunica isso melhor.

### Item 3 — Redirecionamento para bootstrap

**Prompt:** "Refaz usando `scipy.stats.bootstrap` pra calcular IC 95% da satisfação média de cada canal."

In [ ]:
# CÓDIGO GERADO PELO LLM (após redirecionamento) — executado e revisado pelo aluno
from scipy import stats

df_sat = df_consolidado.dropna(subset=["satisfacao_cliente"])

chat_sat  = df_sat.loc[df_sat["canal"] == "chat",  "satisfacao_cliente"].values
email_sat = df_sat.loc[df_sat["canal"] == "email", "satisfacao_cliente"].values

rng_bs = np.random.default_rng(42)

bs_chat = stats.bootstrap(
    (chat_sat,),
    statistic=np.mean,
    confidence_level=0.95,
    n_resamples=9999,
    random_state=rng_bs,
    method="percentile",
)

bs_email = stats.bootstrap(
    (email_sat,),
    statistic=np.mean,
    confidence_level=0.95,
    n_resamples=9999,
    random_state=rng_bs,
    method="percentile",
)

print(f"Chat  — média: {chat_sat.mean():.4f} | IC 95%: [{bs_chat.confidence_interval.low:.4f}, {bs_chat.confidence_interval.high:.4f}]")
print(f"Email — média: {email_sat.mean():.4f} | IC 95%: [{bs_email.confidence_interval.low:.4f}, {bs_email.confidence_interval.high:.4f}]")

### Item 4: Comparação dos intervalos

In [ ]:
# Verificação de sobreposição
ic_chat  = (bs_chat.confidence_interval.low,  bs_chat.confidence_interval.high)
ic_email = (bs_email.confidence_interval.low, bs_email.confidence_interval.high)

sobreposicao = ic_chat[1] >= ic_email[0] and ic_email[1] >= ic_chat[0]
print(f"IC Chat:  [{ic_chat[0]:.4f}, {ic_chat[1]:.4f}]")
print(f"IC Email: [{ic_email[0]:.4f}, {ic_email[1]:.4f}]")
print(f"Os intervalos se sobrepõem? {sobreposicao}")

Se os ICs **não se sobrepõem**: a diferença entre chat e email é consistente — difícil ser só variação amostral.  
Se **se sobrepõem**: os dados não são suficientes pra descartar que a diferença seja ruído.

---

## Exercício 6 — Interpretação dos intervalos de confiança

### Item 1 — Interpretação do LLM

**Prompt:** "Interprete os ICs bootstrap de 95% da satisfação média de chat e email."

**LLM respondeu:**  
> "Se os ICs não se sobrepõem, há evidência de diferença com 95% de confiança. O canal influencia a satisfação. Se a diferença for ≥ 0,5 pontos na escala 1–5, tem relevância prática e justifica mudança operacional."

### Item 2 — Validação frase por frase

**"Se os ICs não se sobrepõem, há evidência estatística de diferença."**  
 Correto. ICs sem sobreposição indicam distribuições amostrais suficientemente separadas.

**"O canal influencia a satisfação."**  
️ Associação não é causalidade. O canal pode ser proxy do tipo de problema que chega por ele. Os dados mostram correlação, não que o canal *causa* a diferença.

**"Diferença ≥ 0,5 pontos tem relevância prática."**  
 Errado. Esse corte é arbitrário. O que é relevante depende do negócio, não de um número que o LLM inventou.

In [ ]:
diff_media = email_sat.mean() - chat_sat.mean()
print(f"Diferença de satisfação média (email − chat): {diff_media:.3f} pontos")
print(f"Escala de satisfação: 1 a 5")
print(f"Diferença relativa: {diff_media / 4 * 100:.1f}% da amplitude total da escala")

Chat tem satisfação média um pouco abaixo do email, e a diferença aparece de forma consistente nos dados. Mas é pequena em magnitude — antes de mudar qualquer processo, vale investigar por quê: é o tipo de problema que chega por chat, o tempo de resposta, ou a qualidade do atendimento?

---

## Exercício 7 — Visualização com revisão crítica

### Prompt enviado ao LLM

```
Gere com matplotlib um gráfico que compare a satisfação média entre os canais
"chat" e "email" a partir do DataFrame df_consolidado. O gráfico deve responder
à pergunta: "O canal de abertura do ticket influencia a satisfação do cliente?"
```

### Código gerado pelo LLM

In [ ]:
import matplotlib.pyplot as plt

# CÓDIGO GERADO PELO LLM — início
df_canais = df_consolidado[df_consolidado["canal"].isin(["chat", "email"])]
medias = df_canais.groupby("canal", observed=True)["satisfacao_cliente"].mean()

fig, ax = plt.subplots(figsize=(6, 4))
medias.plot(kind="bar", ax=ax, color=["steelblue", "darkorange"])
ax.set_title("Satisfação média por canal")
ax.set_xlabel("Canal")
ax.set_ylabel("Satisfação média")
plt.tight_layout()
plt.savefig("chart_llm.png", dpi=120)
plt.show()
# CÓDIGO GERADO PELO LLM — fim 

### Avaliação do gráfico gerado

**Tipo:** barras — ok pra comparar dois grupos. 

**Eixo Y zerado:** a escala de satisfação vai de 1 a 5. Com eixo em 0, as barras ficam tão parecidas visualmente que a diferença some. ️

**Só a média:** desvio padrão próximo de 1 ponto. Mostrar só a média esconde que os dois canais têm muita sobreposição na distribuição real. 

**Título:** descreve o gráfico em vez de comunicar o achado. 

In [ ]:
# VERSÃO CORRIGIDA PELO ALUNO
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df_canais = df_consolidado[
    df_consolidado["canal"].isin(["chat", "email"]) &
    df_consolidado["satisfacao_cliente"].notna()
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

#  Painel esquerdo: barras com IC bootstrap
canais_ord = ["chat", "email"]
medias_v   = [chat_sat.mean(), email_sat.mean()]
erros_low  = [medias_v[0] - ic_chat[0],  medias_v[1] - ic_email[0]]
erros_high = [ic_chat[1]  - medias_v[0], ic_email[1] - medias_v[1]]

cores = ["#4C72B0", "#DD8452"]
ax0 = axes[0]
bars = ax0.bar(canais_ord, medias_v, color=cores, width=0.5,
               yerr=[erros_low, erros_high], capsize=8, error_kw={"elinewidth": 2})
ax0.set_ylim(3.0, 5.0)  
ax0.set_ylabel("Satisfação média (escala 1–5)")
ax0.set_title("Satisfação média com IC 95% (bootstrap)", fontsize=11)
for bar, m in zip(bars, medias_v):
    ax0.text(bar.get_x() + bar.get_width() / 2, m + 0.05, f"{m:.3f}",
             ha="center", va="bottom", fontsize=10, fontweight="bold")
ax0.set_xlabel("Canal de abertura")

# Painel direito: boxplot 
ax1 = axes[1]
dados_box = [
    df_canais.loc[df_canais["canal"] == "chat",  "satisfacao_cliente"].values,
    df_canais.loc[df_canais["canal"] == "email", "satisfacao_cliente"].values,
]
bp = ax1.boxplot(dados_box, labels=["chat", "email"], patch_artist=True,
                 medianprops={"color": "black", "linewidth": 2})
for patch, cor in zip(bp["boxes"], cores):
    patch.set_facecolor(cor)
    patch.set_alpha(0.7)
ax1.set_ylim(0.5, 5.5)
ax1.set_ylabel("Satisfação do cliente (1–5)")
ax1.set_title("Distribuição da satisfação por canal", fontsize=11)
ax1.set_xlabel("Canal de abertura")

fig.suptitle(
    "Canal de abertura e satisfação do cliente — CloudDesk\n"
    "Chat apresenta satisfação levemente inferior ao e-mail; diferença pequena em magnitude",
    fontsize=12, y=1.02
)
plt.tight_layout()
plt.savefig("chart_corrigido.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico corrigido salvo em chart_corrigido.png")

**Correções feitas:**
1. Eixo Y começa em 3,0 — a diferença real fica visível sem exagerar.
2. Barras de erro com IC 95% bootstrap incluídas.
3. Boxplot ao lado pra mostrar a distribuição completa.
4. Título diz o que o gráfico conclui, não só o que ele mostra.

---

## Exercício 8 — Mini pipeline completo: tickets de "bug" nos 30 dias mais recentes

In [ ]:
# Item 1: Limpeza do recorte com código gerado e revisado via LLM

# Prompt usado: "Filtre df_consolidado para tickets de categoria 'bug'
# nos 30 dias mais recentes cobertos pela base. Mostre o shape e verifique nulos."

# CODIGO GERADO PELO LLM (revisado e executado)
data_max   = df_consolidado["data_abertura"].max()
data_corte = data_max - pd.Timedelta(days=30)

df_bug30 = df_consolidado[
    (df_consolidado["categoria"] == "bug") &
    (df_consolidado["data_abertura"] >= data_corte)
].copy()

print(f"Período: {data_corte.date()} a {data_max.date()}")
print(f"Shape do recorte (bug, 30d): {df_bug30.shape}")
print(f"Nulos por coluna relevante:")
print(df_bug30[["tempo_resposta_horas", "satisfacao_cliente"]].isnull().sum())

### Item 2 — Pergunta analítica

**Prompt:** "Sugere uma pergunta analítica sobre bugs nos últimos 30 dias que compare uma métrica numérica entre dois subgrupos."

**LLM:** *"Tickets de bug com prioridade alta/crítica têm tempo de resolução maior que os de baixa/média?"*

Selecionada — tem implicação direta na priorização do backlog e é respondível com os dados disponíveis.

In [ ]:
# Item 2: Análise descritiva com código gerado pelo LLM
df_bug30["grupo_prioridade"] = df_bug30["prioridade"].map(
    lambda p: "alta/crítica" if str(p) in ["alta", "critica"] else "baixa/média"
)

resumo_bug = (
    df_bug30
    .groupby("grupo_prioridade")["tempo_resolucao_horas"]
    .agg(media="mean", mediana="median", n="count")
    .round(2)
)
print(resumo_bug)

In [ ]:
# Item 3: O LLM sugeriu teste t — redirecionado para bootstrap

# Prompt de redirecionamento:
# "Use scipy.stats.bootstrap para calcular IC 95% do tempo de resolução
#  de cada grupo e compare os intervalos."

grupo_ac  = df_bug30.loc[df_bug30["grupo_prioridade"] == "alta/crítica",  "tempo_resolucao_horas"].values
grupo_bm  = df_bug30.loc[df_bug30["grupo_prioridade"] == "baixa/média",   "tempo_resolucao_horas"].values

rng_bs2 = np.random.default_rng(42)

bs_ac = stats.bootstrap((grupo_ac,), statistic=np.mean, confidence_level=0.95,
                         n_resamples=9999, random_state=rng_bs2, method="percentile")
bs_bm = stats.bootstrap((grupo_bm,), statistic=np.mean, confidence_level=0.95,
                         n_resamples=9999, random_state=rng_bs2, method="percentile")

print(f"Alta/crítica — média: {grupo_ac.mean():.2f}h | IC 95%: [{bs_ac.confidence_interval.low:.2f}, {bs_ac.confidence_interval.high:.2f}]")
print(f"Baixa/média  — média: {grupo_bm.mean():.2f}h | IC 95%: [{bs_bm.confidence_interval.low:.2f}, {bs_bm.confidence_interval.high:.2f}]")

# Sobreposição
sobr = bs_ac.confidence_interval.high >= bs_bm.confidence_interval.low and \
       bs_bm.confidence_interval.high >= bs_ac.confidence_interval.low
print(f"Intervalos se sobrepõem? {sobr}")

### Validação da interpretação do LLM

O LLM disse: *"Se os ICs não se sobrepõem, há diferença relevante no tempo de resolução entre os grupos."*

 Correto sobre sobreposição e o que ela indica.  
️ Com recorte de 30 dias o n é menor, então os ICs são mais largos. Resultado não sobreposição aqui seria evidência forte, mas não garante que vale pro dataset inteiro.

### Item 4 — Avaliação crítica do processo

**Onde a IA foi útil:** código pandas (filtros, groupby, bootstrap) saiu executável direto, sem correção. Acelerou muito a prototipagem.

**Onde a revisão humana foi necessária:** na interpretação dos ICs, o LLM inventou um corte de relevância prática ("0,5 pontos") sem base nenhuma. Sem revisar, esse número entraria como verdade e poderia levar a decisão operacional errada. Quem decide o que é relevante é o negócio, não o modelo.